<a href="https://colab.research.google.com/github/ferdous-gp/colab/blob/main/Intent_based_routing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Testing multilingual-e5-small embedding for decision making

In [ ]:
import numpy as np
import requests
from sentence_transformers import SentenceTransformer

# 1. Initialize the lightweight E5 model
# It maps texts into a 384-dimensional space
model = SentenceTransformer("intfloat/multilingual-e5-small")

# 2. Define Golden Reference Queries for each intent
# Crucial: E5 requires the "query: " prefix for searching/matching!
intent_references = {
    "direct_answer": [
        "query: what is your refund policy?",
        "query: how do I reset my password?",
        "query: where are your office headquarters located?",
        "query: comment réinitialiser mon mot de passe?"  # E5 natively handles multilingual queries
    ],
    "api_call": [
        "query: check the weather in Tokyo",
        "query: what is the current stock price of Apple?",
        "query: track my shipment status ID 12345",
        "query: quel est le temps à Paris aujourd'hui?"
    ],
    "rag_search": [
        "query: what offers in MyGP Today?",
        "query: আজকে বেচা বিক্রির কোনও নির্দেশনা আছে",
        "query: কোন প্যাক এ লাভ বেশি?",
        "query: Rajshahi offer ki?"
    ]
}

# Flatten reference data and prepare for encoding
intent_labels = []
reference_texts = []
for intent, examples in intent_references.items():
    for example in examples:
        intent_labels.append(intent)
        reference_texts.append(example)

# Encode all reference examples once and cache them
# (In production, cache these vectors as an array or use a light database like Chroma/Faiss)
reference_embeddings = model.encode(reference_texts, normalize_embeddings=True)


def classify_intent(user_query: str) -> str:
    """Compare user query against reference intents using cosine similarity."""
    # Prefix the incoming user query as instructed by the E5 protocol
    formatted_query = f"query: {user_query}"
    query_embedding = model.encode([formatted_query], normalize_embeddings=True)

    # Calculate cosine similarity (dot product works since vectors are normalized)
    similarities = np.dot(reference_embeddings, query_embedding.T).flatten()
    best_match_idx = np.argmax(similarities)

    # Optional: Implement a confidence threshold
    if similarities[best_match_idx] < 0.65:
        return "fallback_unknown"

    return intent_labels[best_match_idx]


# 3. Define Action Handlers
def execute_direct_answer(query: str):
    print("🤖 [Routing to Knowledge Base/RAG] Searching vector database for static documentation...")
    # Here you would typically prepend "passage: " to your DB texts, search them, and return text.
    return "Our return policy allows standard claims up to 30 days from purchase."

def execute_api_call(query: str):
    print("🌐 [Routing to API handler] Parsed explicit external demand. Triggering system call...")
    # Example: Mocking a real weather API execution
    # mock_url = f"https://weatherapi.com{query}"
    return "Weather API response: The current temperature in your requested city is 22°C with clear skies."

def execute_rag_search(query: str):
    print("🌐 [Routing to RAG Search] Need to call RAG database for getting answer... ")
    # Example: Mocking a real weather API execution
    # mock_url = f"https://weatherapi.com{query}"
    return "Weather API response: The current temperature in your requested city is 22°C with clear skies."


# 4. Main Chat Engine Orchestrator
def chat_router(user_input: str):
    print(f"\nUser: {user_input}")

    # Step A: Intent Classification
    intent = classify_intent(user_input)
    print(f"Detected Intent: {intent}")

    # Step B: Execution Branching
    if intent == "direct_answer":
        response = execute_direct_answer(user_input)
    elif intent == "api_call":
        response = execute_api_call(user_input)
    elif intent == "rag_search":
        response = execute_rag_search(user_input)
    else:
        response = "I'm sorry, I couldn't determine how to handle that request. Could you rephrase?"

    print(f"Bot: {response}")

# --- Test Runs ---
chat_router("Can I return this item if I don't like it?")
chat_router("Is it raining in London right now?")
chat_router("Dhaka ki offer?")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


User: Can I return this item if I don't like it?
Detected Intent: direct_answer
🤖 [Routing to Knowledge Base/RAG] Searching vector database for static documentation...
Bot: Our return policy allows standard claims up to 30 days from purchase.

User: Is it raining in London right now?
Detected Intent: api_call
🌐 [Routing to API handler] Parsed explicit external demand. Triggering system call...
Bot: Weather API response: The current temperature in your requested city is 22°C with clear skies.

User: Dhaka ki offer?
Detected Intent: rag_search
🌐 [Routing to RAG Search] Need to call RAG database for getting answer... 
Bot: Weather API response: The current temperature in your requested city is 22°C with clear skies.


# Testing with different intents

In [ ]:
chat_router("মাইজিপি তে কোন প্যাক বেচলে বেশি লাভ? ")
chat_router("আজকের দিনের instruction কি")


User: মাইজিপি তে কোন প্যাক বেচলে বেশি লাভ? 
Detected Intent: rag_search
🌐 [Routing to RAG Search] Need to call RAG database for getting answer... 
Bot: Weather API response: The current temperature in your requested city is 22°C with clear skies.

User: আজকের দিনের instruction কি
Detected Intent: rag_search
🌐 [Routing to RAG Search] Need to call RAG database for getting answer... 
Bot: Weather API response: The current temperature in your requested city is 22°C with clear skies.


# Extended Test: Local Vector DB setup for RAG retrival

In [ ]:
import re
import chromadb
from sentence_transformers import SentenceTransformer
import spacy

# Initialize the NLP models
# 1. E5 for structural embeddings
embedding_model = SentenceTransformer("intfloat/multilingual-e5-small")
# 2. SpaCy for extracting Entities (Locations, Dates, etc.)
nlp = spacy.load("en_core_web_sm")

# Initialize Local Vector Database (ChromaDB)
# In-memory for testing; use chromadb.PersistentClient(path="./db") to save to disk
chroma_client = chromadb.Client()


# ==========================================
# BRANCH 1: KNOWLEDGE BASE SETUP (Direct Answers)
# ==========================================
def setup_knowledge_base():
    """Populates the local vector database with reference documents."""
    # Create or get a collection
    collection = chroma_client.get_or_create_collection(name="faq_docs")

    # Sample internal documents
    raw_passages = [
        "Our standard return policy allows items to be returned within 30 days of purchase for a full refund.",
        "To reset your password, click on 'Forgot Password' on the login screen and follow the email verification steps.",
        "Our corporate headquarters is located at 123 Innovation Way, San Francisco, California.",
    ]

    # CRUCIAL: E5 requires documents to be prefixed with "passage: "
    prefixed_passages = [f"passage: {p}" for p in raw_passages]

    # Generate embeddings for the documents
    embeddings = embedding_model.encode(
        prefixed_passages, normalize_embeddings=True
    ).tolist()

    # Add to ChromaDB
    collection.add(
        ids=["doc_refund", "doc_password", "doc_hq"],
        embeddings=embeddings,
        documents=raw_passages,  # Store raw text to return to the user
    )
    print("✅ Knowledge Base Initialized.")


def execute_direct_answer(user_query: str) -> str:
    """Searches the vector database for a matching document passage."""
    collection = chroma_client.get_collection(name="faq_docs")

    # CRUCIAL: Prepend "query: " to the search phrase
    formatted_query = f"query: {user_query}"
    query_vector = embedding_model.encode(
        [formatted_query], normalize_embeddings=True
    ).tolist()

    # Query the database for the top match
    results = collection.query(
        query_embeddings=query_vector, n_results=1, include=["documents", "distances"]
    )

    # Check if a document was returned and look at the distance score
    if results["documents"] and results["documents"][0]:
        # ChromaDB returns L2 distance; lower means closer match
        distance = results["distances"][0][0]
        if distance < 0.5:  # Adjust threshold depending on your data
            return results["documents"][0][0]

    return "I found some documentation, but it might not be relevant enough to answer your question."


# ==========================================
# BRANCH 2: ARGUMENT EXTRACTION (API Calls)
# ==========================================
def extract_api_parameters(user_query: str) -> dict:
    """Extracts variables needed for API calls using NLP and Regular Expressions."""
    params = {"location": None, "tracking_id": None}

    # 1. Extract Locations (GPE: Geopolitical Entity) using SpaCy NER
    doc = nlp(user_query)
    for ent in doc.ents:
        if ent.label_ in ["GPE", "LOC"]:
            params["location"] = ent.text
            break

    # 2. Extract structured codes (like an ID number) using Regex
    # Matches strings like "ID 12345" or just a 5+ digit numeric sequence
    id_match = re.search(r"\b(?:id\s*|#)?(\d{5,})\b", user_query, re.IGNORECASE)
    if id_match:
        params["tracking_id"] = id_match.group(1)

    return params


def execute_api_call(user_query: str) -> str:
    """Extracts arguments and executes the target external system request."""
    # Pull needed arguments out of the string
    extracted_args = extract_api_parameters(user_query)

    # Route based on what context clues we have
    if "weather" in user_query.lower() or "temperature" in user_query.lower():
        city = extracted_args["location"]
        if not city:
            return "I recognize you want the weather, but which city are you asking about?"

        # Example Mock: Replace this block with a real `requests.get()` to a Weather API
        return f"🌐 API TRIGGERED: Fetching current conditions for [{city}]. Result: 18°C, Partly Cloudy."

    elif (
        "track" in user_query.lower()
        or "shipment" in user_query.lower()
        or "package" in user_query.lower()
    ):
        package_id = extracted_args["tracking_id"]
        if not package_id:
            return "Please provide a valid numeric shipment tracking ID."

        # Example Mock: Replace this block with a real logistics API call
        return f"📦 API TRIGGERED: Querying logistics carrier for ID [{package_id}]. Result: In Transit - Out for delivery."

    return "Intent mapped to API system, but no specific action or parameters matched."


# ==========================================
# 🚀 EXECUTION DEMO
# ==========================================
# 1. Run the database setup
setup_knowledge_base()

print("\n--- Testing Direct Answer Branch ---")
q1 = "Can I get my money back if I return a product?"
print(f"Query: {q1}\nResponse: {execute_direct_answer(q1)}")

print("\n--- Testing API Extraction Branch (Weather) ---")
q2 = "What is the temperature like in London tomorrow morning?"
print(f"Query: {q2}\nResponse: {execute_api_call(q2)}")

print("\n--- Testing API Extraction Branch (Tracking) ---")
q3 = "Where is my package? The reference code is 987654"
print(f"Query: {q3}\nResponse: {execute_api_call(q3)}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Knowledge Base Initialized.

--- Testing Direct Answer Branch ---
Query: Can I get my money back if I return a product?
Response: Our standard return policy allows items to be returned within 30 days of purchase for a full refund.

--- Testing API Extraction Branch (Weather) ---
Query: What is the temperature like in London tomorrow morning?
Response: 🌐 API TRIGGERED: Fetching current conditions for [London]. Result: 18°C, Partly Cloudy.

--- Testing API Extraction Branch (Tracking) ---
Query: Where is my package? The reference code is 987654
Response: 📦 API TRIGGERED: Querying logistics carrier for ID [987654]. Result: In Transit - Out for delivery.
